# Lab Day 1 — Neural Network and Training Experiments

Notebook đã được hoàn thiện từ khung `lab.ipynb`.

**Mục tiêu:** hoàn thiện Part 0 → Part 4, chạy baseline, thực hiện một số thí nghiệm có kiểm soát, chọn cấu hình bằng validation và đánh giá cuối trên `eval`.

> Notebook này không dùng pretrained model, `MLPClassifier` hay softmax trong model. Mọi lựa chọn model cuối đều dựa trên validation; `eval` chỉ dùng ở bước cuối.

In [ ]:
# ===== 0. Cấu hình môi trường =====
import os, sys, json, time, math, random, subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, precision_recall_fscore_support

# Notebook nằm ở: submission_<MSSV>/code/lab.ipynb
REPO_ROOT = Path("../..").resolve()
OUT_DIR = Path("..").resolve()

FIG_DIR = OUT_DIR / "figures"
RES_DIR = OUT_DIR / "results"
FIG_DIR.mkdir(parents=True, exist_ok=True)
RES_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
VAL_FRACTION = 0.20

if torch.cuda.is_available():
    device = torch.device("cuda")
    device_name = torch.cuda.get_device_name(0)
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
    device_name = "Apple MPS"
else:
    device = torch.device("cpu")
    device_name = "CPU"

print("PyTorch:", torch.__version__)
print("Device:", device)
print("Device name:", device_name)
print("Repo root:", REPO_ROOT)
print("Output:", OUT_DIR)

## Part 0 — Chuẩn bị dữ liệu

Tập `train` được tách tiếp thành train/validation theo seed 42. Chỉ 10 feature liên tục đầu tiên được chuẩn hóa; 44 feature nhị phân giữ nguyên. `eval` không được dùng để chọn cấu hình.

In [ ]:
# ===== Part 0: seed + split data + normalization =====
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def _find_key(npz, candidates):
    for k in candidates:
        if k in npz.files:
            return k
    return None

def load_npz(path):
    z = np.load(path)
    xk = _find_key(z, ["X", "x", "features", "data"])
    yk = _find_key(z, ["y", "Y", "labels", "label", "target"])
    rk = _find_key(z, ["row_id", "row_ids", "ids", "id"])
    if xk is None or yk is None:
        raise KeyError(f"Không tìm thấy X/y trong {path}. Keys={z.files}")
    X = z[xk].astype(np.float32)
    y = z[yk].astype(np.int64)
    row_id = z[rk] if rk is not None else np.arange(len(y), dtype=np.int64)
    return X, y, row_id

def prepare_data(processed_dir, device, val_fraction=0.20, seed=42):
    processed_dir = Path(processed_dir)
    train_path = processed_dir / "train.npz"
    eval_path = processed_dir / "eval.npz"

    if not train_path.exists() or not eval_path.exists():
        raise FileNotFoundError(
            "Chưa có train.npz/eval.npz. Hãy chạy `python scripts/split_data.py` "
            f"từ repo root: {REPO_ROOT}"
        )

    X_all, y_all, _ = load_npz(train_path)
    X_eval, y_eval, row_eval = load_npz(eval_path)

    idx = np.arange(len(y_all))
    tr_idx, val_idx = train_test_split(
        idx, test_size=val_fraction, random_state=seed, stratify=y_all
    )

    X_tr_np, y_tr_np = X_all[tr_idx], y_all[tr_idx]
    X_val_np, y_val_np = X_all[val_idx], y_all[val_idx]

    # Chỉ chuẩn hóa 10 cột liên tục, thống kê lấy từ train thật sự.
    mean10 = X_tr_np[:, :10].mean(axis=0)
    std10 = X_tr_np[:, :10].std(axis=0)
    std10 = np.where(std10 < 1e-12, 1.0, std10)

    X_tr_np = X_tr_np.copy()
    X_val_np = X_val_np.copy()
    X_eval = X_eval.copy()

    X_tr_np[:, :10] = (X_tr_np[:, :10] - mean10) / std10
    X_val_np[:, :10] = (X_val_np[:, :10] - mean10) / std10
    X_eval[:, :10] = (X_eval[:, :10] - mean10) / std10

    data = {
        "X_tr": torch.from_numpy(X_tr_np).to(device),
        "y_tr": torch.from_numpy(y_tr_np).to(device),
        "X_val": torch.from_numpy(X_val_np).to(device),
        "y_val": torch.from_numpy(y_val_np).to(device),
        "X_eval": torch.from_numpy(X_eval).to(device),
        "y_eval": torch.from_numpy(y_eval).to(device),
        "eval_row_id": row_eval,
        "mean10": mean10,
        "std10": std10,
    }
    return data

# Tạo split cố định theo metadata.
split_script = REPO_ROOT / "scripts" / "split_data.py"
if not (REPO_ROOT / "data" / "processed" / "train.npz").exists():
    subprocess.run([sys.executable, str(split_script)], cwd=str(REPO_ROOT), check=True)

data = prepare_data(REPO_ROOT / "data" / "processed", device, VAL_FRACTION, SEED)

print("train:", tuple(data["X_tr"].shape), tuple(data["y_tr"].shape))
print("val:  ", tuple(data["X_val"].shape), tuple(data["y_val"].shape))
print("eval: ", tuple(data["X_eval"].shape), tuple(data["y_eval"].shape))

majority_class = torch.bincount(data["y_val"]).argmax().item()
majority_acc = (data["y_val"] == majority_class).float().mean().item()
print(f"Majority-class baseline: class={majority_class}, val accuracy={majority_acc:.4f}")

print("Mean first 10 train features:", data["X_tr"][:, :10].mean(dim=0).detach().cpu().numpy())
print("Std  first 10 train features:", data["X_tr"][:, :10].std(dim=0, unbiased=False).detach().cpu().numpy())

## Part 1 — Model và kiểm tra sức khỏe ban đầu

Baseline `M-base`:

`54 → 256 → 128 → 7`, ReLU, He initialization, không softmax trong model.

In [ ]:
# ===== Part 1: MLP =====
EXPECTED_PARAMS = {
    (256, 128): 47879,
    (512, 256): 161287,
    (256, 128, 64): 55687,
}

class MLP(nn.Module):
    def __init__(self, hidden=(256, 128), dropout=0.0, init="he"):
        super().__init__()
        layers = []
        in_dim = 54
        self.linear_layers = []

        for h in hidden:
            layer = nn.Linear(in_dim, h)
            self.linear_layers.append(layer)
            layers.extend([layer, nn.ReLU()])
            if dropout and dropout > 0:
                layers.append(nn.Dropout(dropout))
            in_dim = h

        out = nn.Linear(in_dim, 7)
        self.linear_layers.append(out)
        layers.append(out)
        self.net = nn.Sequential(*layers)

        self.init_name = init
        self._initialize(init)

    def _initialize(self, init):
        for layer in self.linear_layers:
            if init == "he":
                nn.init.kaiming_normal_(layer.weight, nonlinearity="relu")
            elif init == "xavier":
                nn.init.xavier_normal_(layer.weight)
            elif init == "normal":
                nn.init.normal_(layer.weight, std=0.01)
            elif init == "zeros":
                nn.init.zeros_(layer.weight)
            else:
                raise ValueError(f"Unknown init={init}")
            nn.init.zeros_(layer.bias)

    def forward(self, x):
        return self.net(x)

def count_params(model):
    return sum(p.numel() for p in model.parameters())

def activation_stats(model, x):
    stats = {}
    hooks = []
    i = 0
    def hook(name):
        def _h(module, inp, out):
            stats[name] = {
                "mean": float(out.detach().mean().cpu()),
                "std": float(out.detach().std().cpu()),
            }
        return _h
    for m in model.modules():
        if isinstance(m, nn.ReLU):
            hooks.append(m.register_forward_hook(hook(f"relu_{i}")))
            i += 1
    with torch.no_grad():
        model(x)
    for h in hooks:
        h.remove()
    return stats

set_seed(SEED)
model = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
assert count_params(model) == EXPECTED_PARAMS[(256, 128)]
print("Parameter count:", count_params(model))

x8 = torch.randn(8, 54, device=device)
logits = model(x8)
print("Input shape:", x8.shape)
print("Output shape:", logits.shape)
assert logits.shape == (8, 7)

model.eval()
with torch.no_grad():
    loss0 = F.cross_entropy(model(data["X_val"]), data["y_val"]).item()
print(f"Loss step 0 = {loss0:.4f}; reference ln(7) = {math.log(7):.4f}")

print("Activation stats:", activation_stats(model, data["X_val"][:1024]))

In [ ]:
# ===== Part 1: overfit 20 samples + gradient check =====
set_seed(SEED)
tiny_model = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
tiny_opt = torch.optim.SGD(tiny_model.parameters(), lr=0.05, momentum=0.9)

tiny_x = data["X_tr"][:20]
tiny_y = data["y_tr"][:20]
tiny_losses = []

tiny_model.train()
for step in range(400):
    tiny_opt.zero_grad(set_to_none=True)
    loss = F.cross_entropy(tiny_model(tiny_x), tiny_y)
    loss.backward()
    tiny_opt.step()
    tiny_losses.append(loss.item())

print(f"20-sample final loss: {tiny_losses[-1]:.6f}")
print(f"20-sample final accuracy: {(tiny_model(tiny_x).argmax(1) == tiny_y).float().mean().item():.3f}")

plt.figure(figsize=(7,4))
plt.plot(tiny_losses)
plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("Part 1 — Overfit 20 samples")
plt.grid(alpha=0.2)
plt.show()

# Gradient check
check_model = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
check_model.train()
check_model.zero_grad(set_to_none=True)
check_loss = F.cross_entropy(check_model(data["X_tr"][:128]), data["y_tr"][:128])
check_loss.backward()

for name, p in check_model.named_parameters():
    norm = None if p.grad is None else float(p.grad.detach().norm().cpu())
    print(f"{name:25s} grad_norm={norm}")
    assert p.grad is not None and torch.isfinite(p.grad).all() and p.grad.norm().item() > 0

### Nhận xét Part 1

Loss bước 0 và bài kiểm tra overfit ở trên là hai sanity check chính. Nếu loss bước 0 gần `ln(7)` và 20 mẫu có thể overfit gần 100%, pipeline model/loss/gradient cơ bản đang hoạt động.

## Part 2 — Pipeline huấn luyện và baseline

Baseline bắt buộc: M-base + He + Cross-Entropy + SGD(momentum=0.9) + batch 512 + 20 epoch + FP32 + không dropout/clipping.

Các thí nghiệm dùng cùng validation split. `train_loss` được đo ở `eval()` sau mỗi epoch để so sánh công bằng với `val_loss`.

In [ ]:
# ===== Part 2: training pipeline =====
DEFAULT_CFG = {
    "hidden": (256, 128),
    "dropout": 0.0,
    "init": "he",
    "loss": "ce",
    "optimizer": "sgd",
    "lr": 0.05,
    "momentum": 0.9,
    "weight_decay": 0.0,
    "batch_size": 512,
    "epochs": 20,
    "clip_norm": None,
    "precision": "fp32",
    "seed": 42,
    "exp_id": "base-s42",
    "group": "baseline",
    "description": "M-base baseline",
}

def build_optimizer(model, cfg):
    name = cfg["optimizer"].lower()
    if name == "sgd":
        return torch.optim.SGD(
            model.parameters(), lr=cfg["lr"],
            momentum=cfg.get("momentum", 0.0),
            weight_decay=cfg.get("weight_decay", 0.0)
        )
    if name == "adam":
        return torch.optim.Adam(
            model.parameters(), lr=cfg["lr"],
            weight_decay=cfg.get("weight_decay", 0.0)
        )
    if name == "adamw":
        return torch.optim.AdamW(
            model.parameters(), lr=cfg["lr"],
            weight_decay=cfg.get("weight_decay", 0.0)
        )
    raise ValueError(f"Unknown optimizer: {name}")

def _loss_fn(logits, y, loss_name):
    if loss_name == "ce":
        return F.cross_entropy(logits, y)
    if loss_name == "mse":
        target = F.one_hot(y, num_classes=7).float()
        return F.mse_loss(torch.softmax(logits, dim=1), target)
    raise ValueError(loss_name)

@torch.no_grad()
def evaluate(model, X, y, loss_name="ce", batch_size=8192):
    model.eval()
    losses, preds = [], []
    for start in range(0, len(y), batch_size):
        xb = X[start:start+batch_size]
        yb = y[start:start+batch_size]
        logits = model(xb)
        losses.append(_loss_fn(logits, yb, loss_name).item() * len(yb))
        preds.append(logits.argmax(dim=1).cpu().numpy())
    pred = np.concatenate(preds)
    yy = y.cpu().numpy()
    return {
        "loss": float(sum(losses) / len(y)),
        "acc": float(accuracy_score(yy, pred)),
        "macro_f1": float(f1_score(yy, pred, average="macro", zero_division=0)),
        "pred": pred,
    }

def run_experiment(cfg, data):
    set_seed(cfg["seed"])
    model = MLP(cfg["hidden"], cfg["dropout"], cfg["init"]).to(device)
    optimizer = build_optimizer(model, cfg)

    # Loss step 0 before first update.
    initial = evaluate(model, data["X_val"], data["y_val"], cfg["loss"])
    history = {
        "epoch": [],
        "train_loss": [],
        "val_loss": [],
        "val_acc": [],
        "val_macro_f1": [],
        "grad_norm": [],
        "epoch_seconds": [],
        "max_memory_mb": [],
    }

    Xtr, ytr = data["X_tr"], data["y_tr"]
    n = len(ytr)
    diverged = False
    best_state = None
    best_val_loss = float("inf")
    best_epoch = 0

    for epoch in range(1, cfg["epochs"] + 1):
        model.train()
        t0 = time.perf_counter()
        perm = torch.randperm(n, device=device)

        grad_norms = []
        for start in range(0, n, cfg["batch_size"]):
            idx = perm[start:start+cfg["batch_size"]]
            xb, yb = Xtr[idx], ytr[idx]

            optimizer.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = _loss_fn(logits, yb, cfg["loss"])

            if not torch.isfinite(loss):
                diverged = True
                break

            loss.backward()

            # Measure before clipping.
            total_sq = 0.0
            for p in model.parameters():
                if p.grad is not None:
                    total_sq += float(p.grad.detach().norm(2).cpu()) ** 2
            grad_norms.append(math.sqrt(total_sq))

            if cfg.get("clip_norm") is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["clip_norm"])

            optimizer.step()

        if torch.cuda.is_available():
            torch.cuda.synchronize()
        elapsed = time.perf_counter() - t0

        if diverged:
            break

        tr = evaluate(model, Xtr, ytr, cfg["loss"])
        va = evaluate(model, data["X_val"], data["y_val"], cfg["loss"])

        history["epoch"].append(epoch)
        history["train_loss"].append(tr["loss"])
        history["val_loss"].append(va["loss"])
        history["val_acc"].append(va["acc"])
        history["val_macro_f1"].append(va["macro_f1"])
        history["grad_norm"].append(float(np.mean(grad_norms)))
        history["epoch_seconds"].append(elapsed)

        if torch.cuda.is_available():
            history["max_memory_mb"].append(
                torch.cuda.max_memory_allocated() / 1024**2
            )
            torch.cuda.reset_peak_memory_stats()
        else:
            history["max_memory_mb"].append(0.0)

        if va["loss"] < best_val_loss:
            best_val_loss = va["loss"]
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        print(
            f"{cfg['exp_id']} | epoch {epoch:02d}/{cfg['epochs']} | "
            f"train_loss={tr['loss']:.4f} | val_loss={va['loss']:.4f} | "
            f"val_acc={va['acc']:.4f} | val_macro_f1={va['macro_f1']:.4f}"
        )

    # Restore best validation-loss state.
    if best_state is not None:
        model.load_state_dict(best_state)

    best_val = evaluate(model, data["X_val"], data["y_val"], cfg["loss"])
    result = {
        "exp_id": cfg["exp_id"],
        "group": cfg.get("group", ""),
        "description": cfg.get("description", ""),
        "cfg": dict(cfg),
        "history": history,
        "initial_val_loss": initial["loss"],
        "best_epoch": best_epoch,
        "best_val_loss": best_val["loss"],
        "best_val_acc": best_val["acc"],
        "best_val_macro_f1": best_val["macro_f1"],
        "diverged": diverged,
        "model_state": best_state,
    }
    return result

def save_result(result, folder=RES_DIR):
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    clean = {k:v for k,v in result.items() if k != "model_state"}
    with open(folder / f"{result['exp_id']}.json", "w", encoding="utf-8") as f:
        json.dump(clean, f, ensure_ascii=False, indent=2)

def plot_run(result, path):
    h = result["history"]
    fig, axes = plt.subplots(3, 1, figsize=(9, 11))
    axes[0].plot(h["epoch"], h["train_loss"], label="train")
    axes[0].plot(h["epoch"], h["val_loss"], label="val")
    axes[0].set_title(f"{result['exp_id']} — loss")
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss"); axes[0].legend()

    axes[1].plot(h["epoch"], h["val_acc"], label="val accuracy")
    axes[1].plot(h["epoch"], h["val_macro_f1"], label="val macro-F1")
    axes[1].set_title("Validation metrics")
    axes[1].set_xlabel("Epoch"); axes[1].legend()

    axes[2].plot(h["epoch"], h["grad_norm"])
    axes[2].set_title("Gradient norm (before clipping)")
    axes[2].set_xlabel("Epoch"); axes[2].set_ylabel("L2 norm")

    fig.suptitle(f"{result['exp_id']} | best epoch={result['best_epoch']}")
    fig.tight_layout()
    fig.savefig(path, dpi=140, bbox_inches="tight")
    plt.close(fig)

def plot_compare(results, metric, path):
    plt.figure(figsize=(9,5))
    for r in results:
        plt.plot(r["history"]["epoch"], r["history"][metric], label=r["exp_id"])
    plt.xlabel("Epoch"); plt.ylabel(metric); plt.title(f"Compare — {metric}")
    plt.legend()
    plt.grid(alpha=0.2)
    plt.tight_layout()
    plt.savefig(path, dpi=140, bbox_inches="tight")
    plt.show()

def predict(model, X, batch_size=8192):
    model.eval()
    preds = []
    with torch.no_grad():
        for start in range(0, len(X), batch_size):
            preds.append(model(X[start:start+batch_size]).argmax(1).cpu().numpy())
    return np.concatenate(preds)

def final_eval(result, data, csv_path):
    model = MLP(
        hidden=tuple(result["cfg"]["hidden"]),
        dropout=result["cfg"]["dropout"],
        init=result["cfg"]["init"]
    ).to(device)
    model.load_state_dict(result["model_state"])
    model.eval()
    pred = predict(model, data["X_eval"])
    out = pd.DataFrame({"row_id": data["eval_row_id"], "pred": pred.astype(np.int64)})
    out.to_csv(csv_path, index=False)
    return model, out

In [ ]:
# ===== Part 2: chọn lr bằng validation =====
lr_candidates = [0.01, 0.03, 0.05, 0.1]
lr_results = []

for lr in lr_candidates:
    cfg = {**DEFAULT_CFG, "lr": lr, "epochs": 5,
           "seed": SEED, "exp_id": f"lr-{lr:g}", "group": "lr_search"}
    r = run_experiment(cfg, data)
    lr_results.append(r)

lr_table = pd.DataFrame([
    {"lr": r["cfg"]["lr"], "val_acc": r["best_val_acc"],
     "val_macro_f1": r["best_val_macro_f1"], "best_epoch": r["best_epoch"]}
    for r in lr_results
]).sort_values("val_macro_f1", ascending=False)

display(lr_table)
BEST_LR = float(lr_table.iloc[0]["lr"])
print("Selected LR by validation:", BEST_LR)

In [ ]:
# ===== Part 2: baseline, 3 seeds =====
baseline_results = []
baseline_seeds = [42, 43, 44]

for s in baseline_seeds:
    cfg = {**DEFAULT_CFG, "lr": BEST_LR, "seed": s,
           "exp_id": f"base-s{s}", "group": "baseline",
           "description": "M-base / SGD+momentum / He / CE"}
    r = run_experiment(cfg, data)
    baseline_results.append(r)
    save_result(r)
    plot_run(r, FIG_DIR / f"{r['exp_id']}.png")

seed_df = pd.DataFrame([
    {"exp_id": r["exp_id"], "val_acc": r["best_val_acc"],
     "val_macro_f1": r["best_val_macro_f1"]}
    for r in baseline_results
])
display(seed_df)

baseline_mean = seed_df["val_macro_f1"].mean()
baseline_std = seed_df["val_macro_f1"].std(ddof=1)
noise_2sigma = 2 * baseline_std
print(f"Baseline val macro-F1: {baseline_mean:.4f} ± {baseline_std:.4f}")
print(f"2σ noise threshold: {noise_2sigma:.4f}")

plot_compare(baseline_results, "val_macro_f1", FIG_DIR / "compare_baseline_seeds.png")

## Part 3 — Thí nghiệm

Ba thí nghiệm mặc định dưới đây chỉ đổi **một yếu tố** so với baseline:

1. Adam optimizer
2. Dropout 0.3
3. Gradient clipping

Có thể mở rộng thêm các nhóm loss, hyperparameter, mixed precision hoặc initialization nếu cần. Không dùng `eval` để chọn.

In [ ]:
# ===== Part 3: controlled experiments =====
baseline_cfg = {**DEFAULT_CFG, "lr": BEST_LR, "seed": 42}

experiment_cfgs = [
    {
        **baseline_cfg,
        "exp_id": "opt-adam",
        "group": "optimizer",
        "optimizer": "adam",
        "lr": BEST_LR,
        "description": "Change optimizer only: SGD+momentum -> Adam",
    },
    {
        **baseline_cfg,
        "exp_id": "drop-0.3",
        "group": "dropout",
        "dropout": 0.3,
        "description": "Change dropout only: q=0 -> q=0.3",
    },
    {
        **baseline_cfg,
        "exp_id": "clip-1.0",
        "group": "clipping",
        "clip_norm": 1.0,
        "description": "Change gradient clipping only: no clip -> max norm 1.0",
    },
]

predictions = {
    "opt-adam": "Adam có thể hội tụ nhanh hơn SGD do thích nghi theo gradient từng tham số.",
    "drop-0.3": "Nếu baseline chưa quá khớp, dropout có thể không cải thiện đáng kể; nếu có overfit, khoảng cách train/val có thể giảm.",
    "clip-1.0": "Ở learning rate baseline, clipping có thể ít ảnh hưởng nếu gradient hiếm khi vượt ngưỡng.",
}

experiment_results = []
for cfg in experiment_cfgs:
    print("\n" + "="*80)
    print(cfg["exp_id"])
    print("Dự đoán trước:", predictions[cfg["exp_id"]])
    r = run_experiment(cfg, data)
    experiment_results.append(r)
    save_result(r)
    plot_run(r, FIG_DIR / f"{r['exp_id']}.png")
    diff = r["best_val_macro_f1"] - baseline_mean
    print(f"Kết quả: val_macro_f1={r['best_val_macro_f1']:.4f}; "
          f"baseline_mean={baseline_mean:.4f}; diff={diff:+.4f}; "
          f"vượt 2σ? {abs(diff) > noise_2sigma}")

summary = pd.DataFrame([
    {"exp_id": "baseline_mean", "group": "baseline",
     "val_acc": seed_df["val_acc"].mean(),
     "val_macro_f1": seed_df["val_macro_f1"].mean()}
] + [
    {"exp_id": r["exp_id"], "group": r["group"],
     "val_acc": r["best_val_acc"], "val_macro_f1": r["best_val_macro_f1"]}
    for r in experiment_results
])
display(summary)

In [ ]:
# ===== Part 3: comparison figures by group =====
groups = {}
for r in experiment_results:
    groups.setdefault(r["group"], []).append(r)

for group, rs in groups.items():
    plot_compare(rs, "val_loss", FIG_DIR / f"compare_{group}.png")

### Đối chiếu sau thí nghiệm

Bảng dưới đây giúp kiểm tra xem chênh lệch có lớn hơn nhiễu 2σ của baseline hay không. Kết luận cuối cùng phải dựa trên số liệu này, không chỉ dựa vào cảm giác từ đường cong.

In [ ]:
# ===== Part 4: chọn cấu hình cuối bằng validation =====
all_candidates = [
    max(baseline_results, key=lambda r: r["best_val_macro_f1"])
] + experiment_results

final_result = max(all_candidates, key=lambda r: r["best_val_macro_f1"])
print("Selected final configuration by VAL only:")
print(final_result["exp_id"])
print(json.dumps({k:v for k,v in final_result["cfg"].items()}, indent=2, default=str))
print("Best val macro-F1:", final_result["best_val_macro_f1"])

# Ghi prediction cuối trên toàn bộ eval.
pred_path = OUT_DIR / "predictions_eval.csv"
final_model, predictions_df = final_eval(final_result, data, pred_path)

print("Saved:", pred_path)
print("Rows:", len(predictions_df))
print(predictions_df.head())
assert len(predictions_df) == 116203
assert predictions_df["row_id"].nunique() == len(predictions_df)
assert predictions_df["pred"].between(0, 6).all()

In [ ]:
# ===== Part 4: official evaluation for baseline and final =====
evaluate_script = REPO_ROOT / "scripts" / "evaluate.py"

# Baseline: chọn seed baseline tốt nhất theo validation; eval chỉ được chạy để báo cáo.
best_baseline = max(baseline_results, key=lambda r: r["best_val_macro_f1"])
baseline_pred_tmp = OUT_DIR / "_baseline_predictions_eval.csv"
_, baseline_pred_df = final_eval(best_baseline, data, baseline_pred_tmp)
baseline_eval_path = OUT_DIR / "_baseline_eval_result.json"

cmd_base = [
    sys.executable, str(evaluate_script),
    "--pred", str(baseline_pred_tmp),
    "--out", str(baseline_eval_path),
]
subprocess.run(cmd_base, cwd=str(REPO_ROOT), check=True)

with open(baseline_eval_path, "r", encoding="utf-8") as f:
    baseline_eval_result = json.load(f)

# Final configuration: đây là file prediction được nộp.
pred_path = OUT_DIR / "predictions_eval.csv"
final_model, predictions_df = final_eval(final_result, data, pred_path)
eval_result_path = OUT_DIR / "eval_result.json"

cmd_final = [
    sys.executable, str(evaluate_script),
    "--pred", str(pred_path),
    "--out", str(eval_result_path),
]
subprocess.run(cmd_final, cwd=str(REPO_ROOT), check=True)

with open(eval_result_path, "r", encoding="utf-8") as f:
    eval_result = json.load(f)

def get_metric(obj, *keys):
    for k in keys:
        if k in obj:
            return obj[k]
    return None

baseline_eval_acc = get_metric(baseline_eval_result, "accuracy", "acc")
baseline_eval_f1 = get_metric(baseline_eval_result, "macro_f1", "macro-F1", "macro_f1_score")
eval_acc = get_metric(eval_result, "accuracy", "acc")
eval_f1 = get_metric(eval_result, "macro_f1", "macro-F1", "macro_f1_score")

print("Baseline eval:", baseline_eval_acc, baseline_eval_f1)
print("Final eval:", eval_acc, eval_f1)
print("Final predictions:", pred_path)

# Temporary baseline eval artifacts are not part of the submission.
for p in [baseline_pred_tmp, baseline_eval_path]:
    try:
        p.unlink()
    except FileNotFoundError:
        pass

In [ ]:
# ===== Part 4: error analysis =====
# Hỗ trợ cả key naming phổ biến từ evaluate.py.
print("Eval result keys:", list(eval_result.keys()))

def get_metric(obj, *keys):
    for k in keys:
        if k in obj:
            return obj[k]
    return None

eval_acc = get_metric(eval_result, "accuracy", "acc")
eval_f1 = get_metric(eval_result, "macro_f1", "macro-F1", "macro_f1_score")
print("Final eval accuracy:", eval_acc)
print("Final eval macro-F1:", eval_f1)

if "per_class" in eval_result:
    display(pd.DataFrame(eval_result["per_class"]))
elif "class_metrics" in eval_result:
    display(pd.DataFrame(eval_result["class_metrics"]))

cm = eval_result.get("confusion_matrix")
if cm is not None:
    cm = np.asarray(cm)
    print("Confusion matrix shape:", cm.shape)
    display(pd.DataFrame(cm, index=[f"true_{i}" for i in range(7)],
                         columns=[f"pred_{i}" for i in range(7)]))

## Part 4 — Tạo `experiments.xlsx`

Bảng được tạo từ các kết quả đã lưu. `eval_acc` và `eval_macro_f1` chỉ xuất hiện ở cấu hình cuối cùng trong bảng kết quả này; không dùng eval để chọn model.

In [ ]:
# ===== Part 4: tạo experiments.xlsx từ template chính thức =====
from openpyxl import load_workbook

template_path = REPO_ROOT / "templates" / "experiment_table_template.xlsx"
xlsx_path = OUT_DIR / "experiments.xlsx"

if not template_path.exists():
    raise FileNotFoundError(f"Không tìm thấy template: {template_path}")

wb = load_workbook(template_path)

# Không đổi tên sheet/cột. Ghi theo header có sẵn trong template.
ws = wb["Experiments"]
headers = [c.value for c in ws[1]]
header_map = {str(h).strip(): i+1 for i, h in enumerate(headers) if h is not None}

all_results = baseline_results + experiment_results

def cfg_value(r, key):
    v = r["cfg"].get(key, "")
    if isinstance(v, (tuple, list)):
        return "-".join(map(str, v))
    return v

for row_no, r in enumerate(all_results, start=2):
    values = {
        "exp_id": r["exp_id"],
        "group": r["group"],
        "description": r["description"],
        "optimizer": r["cfg"]["optimizer"],
        "lr": r["cfg"]["lr"],
        "batch_size": r["cfg"]["batch_size"],
        "epochs": r["cfg"]["epochs"],
        "hidden": cfg_value(r, "hidden"),
        "dropout": r["cfg"]["dropout"],
        "weight_decay": r["cfg"]["weight_decay"],
        "clip_norm": r["cfg"]["clip_norm"],
        "precision": r["cfg"]["precision"],
        "init": r["cfg"]["init"],
        "seed": r["cfg"]["seed"],
        "best_epoch": r["best_epoch"],
        "best_val_loss": r["best_val_loss"],
        "val_acc": r["best_val_acc"],
        "val_macro_f1": r["best_val_macro_f1"],
        "eval_acc": (
            baseline_eval_acc if r["exp_id"] == best_baseline["exp_id"]
            else eval_acc if r["exp_id"] == final_result["exp_id"] else ""
        ),
        "eval_macro_f1": (
            baseline_eval_f1 if r["exp_id"] == best_baseline["exp_id"]
            else eval_f1 if r["exp_id"] == final_result["exp_id"] else ""
        ),
        "figure_file": f"figures/{r['exp_id']}.png",
        "notes": "",
    }
    for key, value in values.items():
        if key in header_map:
            ws.cell(row=row_no, column=header_map[key], value=value)

# Seeds sheet: giữ tên sheet và ghi baseline seed information.
if "Seeds" in wb.sheetnames:
    ws_seed = wb["Seeds"]
    # append after existing content; do not rename the sheet.
    seed_headers = [c.value for c in ws_seed[1]]
    seed_map = {str(h).strip(): i+1 for i, h in enumerate(seed_headers) if h is not None}
    for row_no, r in enumerate(baseline_results, start=2):
        for key, value in {
            "exp_id": r["exp_id"],
            "val_acc": r["best_val_acc"],
            "val_macro_f1": r["best_val_macro_f1"],
        }.items():
            if key in seed_map:
                ws_seed.cell(row=row_no, column=seed_map[key], value=value)

# Summary: ghi các kết quả chính nếu template có các cột tương ứng.
if "Summary" in wb.sheetnames:
    ws_sum = wb["Summary"]
    sum_headers = [c.value for c in ws_sum[1]]
    sum_map = {str(h).strip(): i+1 for i, h in enumerate(sum_headers) if h is not None}
    summary_values = {
        "baseline_val_macro_f1_mean": baseline_mean,
        "baseline_val_macro_f1_std": baseline_std,
        "noise_2sigma": noise_2sigma,
        "baseline_eval_acc": baseline_eval_acc,
        "baseline_eval_macro_f1": baseline_eval_f1,
        "final_exp_id": final_result["exp_id"],
        "final_eval_acc": eval_acc,
        "final_eval_macro_f1": eval_f1,
    }
    for key, value in summary_values.items():
        if key in sum_map:
            ws_sum.cell(row=2, column=sum_map[key], value=value)

wb.save(xlsx_path)
print("Saved:", xlsx_path)
print("Sheets:", wb.sheetnames)

## Checklist trước khi nộp

- `code/lab.ipynb` chạy được từ đầu đến cuối.
- Không còn `NotImplementedError` trong các module code của submission.
- `predictions_eval.csv` có 116,203 dòng và pass `scripts/evaluate.py`.
- `eval_result.json` được tạo bởi script chính thức.
- Mỗi experiment có một ảnh `figures/<exp_id>.png`.
- `experiments.xlsx` chứa các experiment và seed baseline.
- Báo cáo `REPORT.md` phải dùng số liệu từ bảng/JSON và phân tích cơ chế.
- Không dùng `eval` để chọn cấu hình.
- Không nộp dataset hoặc model weights `.pt/.pth/.ckpt`.